# Notebook 2. Self-Sufficiency Standard Lookup

Build the function that looks up the annual self-sufficiency standard
(SSS) for a given family composition in Montgomery County.

Source: *The Self-Sufficiency Standard for Maryland 2023*  
Dr. Diana Pearce, University of Washington Center for Women's Welfare

The SSS is **not merged** with IPUMS — it's a runtime lookup table.
Given the user's family makeup, we return the annual income they need
to be self-sufficient (no government assistance) in Montgomery County.

In [1]:
import pandas as pd

# Load just the file structure first
xl = pd.ExcelFile("MD2023_SSS.xlsx")
print("Sheet names:")
for sheet in xl.sheet_names:
    print(f"  - {sheet}")

Sheet names:
  - Notes
  - By County
  - By Family
  - SSS
  - SSW


In [2]:
# Load the By Family sheet
df_sss = pd.read_excel("MD2023_SSS.xlsx", sheet_name="By Family")

print(f"Shape: {df_sss.shape}")
print(f"\nColumn names (showing exact spelling and trailing spaces):")
for col in df_sss.columns:
    print(f"  '{col}'")

Shape: (17256, 26)

Column names (showing exact spelling and trailing spaces):
  'Family Type'
  'Adult(s)'
  'Infant(s)'
  'Preschooler(s)'
  'Schoolager(s)'
  'Teenager(s)'
  'State '
  'Year'
  'All Families Table #'
  'County'
  'Housing Costs'
  'Child Care Costs'
  'Food Costs '
  'Transportation Costs'
  'Health Care Costs '
  'Miscellaneous costs'
  'Broadband & Cell Phone'
  'Other Necessities'
  'Taxes'
  'Earned Income Tax Credit (-)'
  'Child Care Tax Credit (-)'
  'Child Tax Credit (-)'
  'Hourly Self-Sufficiency Wage                           '
  'Monthly Self-Sufficiency Wage                           '
  'Annual Self-Sufficiency Wage                           '
  'Emergency Savings'


In [3]:
# First few rows to see what real data looks like
df_sss.head(10)

,Family Type,Adult(s),Infant(s),Preschooler(s),Schoolager(s),Teenager(s),State,Year,All Families Table #,County,...,Broadband & Cell Phone,Other Necessities,Taxes,Earned Income Tax Credit (-),Child Care Tax Credit (-),Child Tax Credit (-),Hourly Self-Sufficiency Wage,Monthly Self-Sufficiency Wage,Annual Self-Sufficiency Wage,Emergency Savings
0,a1i0p0s0t0,1,0,0.0,0.0,0.0,MD,2023,1,Allegany County,...,111.15,163.22,575.25,0.00,0.0,0.00,14.10,2481.87,29782.44,67.27
1,a1i1p0s0t0,1,1,0.0,0.0,0.0,MD,2023,1,Allegany County,...,111.15,335.93,1146.51,0.00,-50.0,-166.67,26.91,4736.24,56834.88,226.76
2,a1i0p1s0t0,1,0,1.0,0.0,0.0,MD,2023,1,Allegany County,...,111.15,323.20,1083.84,0.00,-50.0,-166.67,25.76,4533.52,54402.21,210.51
3,a1i0p0s1t0,1,0,0.0,1.0,0.0,MD,2023,1,Allegany County,...,111.15,301.92,979.04,0.00,-50.0,-166.67,23.83,4194.69,50336.27,183.35
4,a1i0p0s0t1,1,0,0.0,0.0,1.0,MD,2023,1,Allegany County,...,111.15,255.53,690.27,-82.62,0.0,-166.67,19.11,3363.00,40355.98,126.18
5,a1i2p0s0t0,1,2,0.0,0.0,0.0,MD,2023,1,Allegany County,...,111.15,447.28,1553.63,0.00,-100.0,-333.33,34.95,6151.50,73818.05,404.50
6,a1i1p1s0t0,1,1,1.0,0.0,0.0,MD,2023,1,Allegany County,...,111.15,434.50,1490.52,0.00,-100.0,-333.33,33.79,5947.86,71374.34,354.51
7,a1i1p0s1t0,1,1,0.0,1.0,0.0,MD,2023,1,Allegany County,...,111.15,412.85,1383.38,0.00,-100.0,-333.33,31.83,5602.49,67229.93,324.94
8,a1i1p0s0t1,1,1,0.0,0.0,1.0,MD,2023,1,Allegany County,...,111.15,367.42,1158.77,0.00,-100.0,-333.33,27.72,4878.24,58538.86,266.96
9,a1i0p2s0t0,1,0,2.0,0.0,0.0,MD,2023,1,Allegany County,...,111.15,421.73,1427.40,0.00,-100.0,-333.33,32.64,5744.22,68930.62,336.28


## Clean column names

The Excel file has trailing spaces in several column names. Strip them
once at load time so we never have to remember to handle them later.

In [4]:
# Strip whitespace from all column names
df_sss.columns = df_sss.columns.str.strip()

# Verify it worked
print("Cleaned column names:")
for col in df_sss.columns:
    print(f"  '{col}'")

Cleaned column names:
  'Family Type'
  'Adult(s)'
  'Infant(s)'
  'Preschooler(s)'
  'Schoolager(s)'
  'Teenager(s)'
  'State'
  'Year'
  'All Families Table #'
  'County'
  'Housing Costs'
  'Child Care Costs'
  'Food Costs'
  'Transportation Costs'
  'Health Care Costs'
  'Miscellaneous costs'
  'Broadband & Cell Phone'
  'Other Necessities'
  'Taxes'
  'Earned Income Tax Credit (-)'
  'Child Care Tax Credit (-)'
  'Child Tax Credit (-)'
  'Hourly Self-Sufficiency Wage'
  'Monthly Self-Sufficiency Wage'
  'Annual Self-Sufficiency Wage'
  'Emergency Savings'


## Filter to Montgomery County

For this app's scope, we only need Montgomery County. Filtering early
makes everything faster and removes 23 counties of noise.

In [5]:
# Keep only Montgomery County rows
df_mc = df_sss[df_sss["County"] == "Montgomery County"].copy()

print(f"Montgomery County rows: {len(df_mc)}")
print(f"\nUnique adult counts: {sorted(df_mc['Adult(s)'].unique())}")
print(f"Unique infant counts: {sorted(df_mc['Infant(s)'].unique())}")
print(f"Unique preschooler counts: {sorted(df_mc['Preschooler(s)'].unique())}")
print(f"Unique schoolager counts: {sorted(df_mc['Schoolager(s)'].unique())}")
print(f"Unique teenager counts: {sorted(df_mc['Teenager(s)'].unique())}")

Montgomery County rows: 719

Unique adult counts: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Unique infant counts: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Unique preschooler counts: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(nan)]
Unique schoolager counts: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(nan)]
Unique teenager counts: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(nan)]


## The lookup function

Given a family composition, return the annual self-sufficiency wage
for Montgomery County. Returns `None` if the exact combination doesn't
exist in the table.

In [6]:
def get_sss_annual(adults, infants, preschoolers, schoolagers, teenagers,
                   county="Montgomery County"):
    """
    Look up the annual Self-Sufficiency Standard for a family composition.
    
    Parameters
    ----------
    adults, infants, preschoolers, schoolagers, teenagers : int
        Number of household members in each age category.
    county : str, default "Montgomery County"
        Maryland county name (must include the word "County").
    
    Returns
    -------
    float or None
        Annual self-sufficiency wage in dollars, or None if the family
        composition doesn't exist in the SSS table.
    """
    match = df_sss[
        (df_sss["County"] == county)
        & (df_sss["Adult(s)"] == adults)
        & (df_sss["Infant(s)"] == infants)
        & (df_sss["Preschooler(s)"] == preschoolers)
        & (df_sss["Schoolager(s)"] == schoolagers)
        & (df_sss["Teenager(s)"] == teenagers)
    ]
    
    if len(match) == 0:
        return None
    
    return float(match["Annual Self-Sufficiency Wage"].iloc[0])

In [7]:
# Test cases
test_cases = [
    (1, 0, 0, 0, 0, "Single adult, no kids"),
    (1, 1, 0, 0, 0, "Single adult, 1 infant"),
    (2, 0, 0, 0, 0, "2 adults, no kids"),
    (2, 1, 0, 1, 0, "2 adults, 1 infant + 1 schoolager"),
    (1, 0, 0, 0, 0, "Family that may not exist"),  # we'll change this
]

for adults, inf, pre, sch, teen, label in test_cases:
    result = get_sss_annual(adults, inf, pre, sch, teen)
    if result is None:
        print(f"{label}: NOT FOUND")
    else:
        print(f"{label}: ${result:,.0f}/year")

Single adult, no kids: $47,294/year
Single adult, 1 infant: $94,675/year
2 adults, no kids: $64,826/year
2 adults, 1 infant + 1 schoolager: $125,962/year
Family that may not exist: $47,294/year


In [8]:
# Try a family composition that almost certainly doesn't exist
result = get_sss_annual(adults=3, infants=0, preschoolers=0, schoolagers=0, teenagers=0)
print(f"3 adults, no kids: {result}")

3 adults, no kids: 81833.76


In [9]:
# What are the actual boundaries of the SSS table for Montgomery County?
print(f"Adults range:        {sorted(df_mc['Adult(s)'].unique())}")
print(f"Infants range:       {sorted(df_mc['Infant(s)'].unique())}")
print(f"Preschoolers range:  {sorted(df_mc['Preschooler(s)'].unique())}")
print(f"Schoolagers range:   {sorted(df_mc['Schoolager(s)'].unique())}")
print(f"Teenagers range:     {sorted(df_mc['Teenager(s)'].unique())}")
print(f"\nTotal Montgomery County family compositions: {len(df_mc)}")

Adults range:        [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Infants range:       [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Preschoolers range:  [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(nan)]
Schoolagers range:   [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(nan)]
Teenagers range:     [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(nan)]

Total Montgomery County family compositions: 719


## CPI Adjustment: 2023 → 2026

The Self-Sufficiency Standard was published in 2023. Living costs have
risen since then. We adjust forward using the BLS Consumer Price Index
for All Urban Consumers (CPI-U), Washington-Arlington-Alexandria area.

For the dry run, we use a hardcoded multiplier based on published CPI 
data. The full BLS API integration will replace this in the final version.

In [11]:
# CPI adjustment factor: 2023 average → 2026 average (approximate)
# Source: BLS CPI-U, Washington-Arlington-Alexandria DMV area
# Hardcoded for now; will pull from BLS API in final version
CPI_2023_TO_2026 = 1.083   # ~8.3% cumulative inflation 2023→2026

def get_sss_annual_adjusted(adults, infants, preschoolers, schoolagers, 
                             teenagers, county="Montgomery County"):
    """
    Self-sufficiency annual wage adjusted from 2023 to current year.
    
    Returns None if the family composition doesn't exist in the table.
    """
    base_2023 = get_sss_annual(adults, infants, preschoolers, 
                                schoolagers, teenagers, county)
    if base_2023 is None:
        return None
    return base_2023 * CPI_2023_TO_2026

In [12]:
# Compare 2023 vs 2026-adjusted for a few cases
for adults, inf, pre, sch, teen, label in [
    (1, 0, 0, 0, 0, "Single adult"),
    (1, 1, 0, 0, 0, "Single adult + infant"),
    (2, 0, 1, 1, 0, "2 adults + preschooler + schoolager"),
]:
    base = get_sss_annual(adults, inf, pre, sch, teen)
    adjusted = get_sss_annual_adjusted(adults, inf, pre, sch, teen)
    print(f"{label}:")
    print(f"  2023: ${base:,.0f}")
    print(f"  2026: ${adjusted:,.0f}  (+${adjusted-base:,.0f})")
    print()

Single adult:
  2023: $47,294
  2026: $51,219  (+$3,925)

Single adult + infant:
  2023: $94,675
  2026: $102,533  (+$7,858)

2 adults + preschooler + schoolager:
  2023: $122,943
  2026: $133,147  (+$10,204)



In [13]:
# Investigate the NaN rows
nan_rows = df_mc[
    df_mc["Preschooler(s)"].isna() | 
    df_mc["Schoolager(s)"].isna() | 
    df_mc["Teenager(s)"].isna()
]
print(f"Rows with NaN children: {len(nan_rows)}")
print(f"\nSample of these rows:")
print(nan_rows[["Family Type", "Adult(s)", "Infant(s)", "Preschooler(s)", 
                "Schoolager(s)", "Teenager(s)", "Annual Self-Sufficiency Wage"]].head(10))

Rows with NaN children: 89

Sample of these rows:
      Family Type  Adult(s)  Infant(s)  Preschooler(s)  Schoolager(s)  \
10276        a1c7         1          7             NaN            NaN   
10277        a1c8         1          8             NaN            NaN   
10278        a1c9         1          9             NaN            NaN   
10279       a1c10         1         10             NaN            NaN   
10490        a2c7         2          7             NaN            NaN   
10491        a2c8         2          8             NaN            NaN   
10492        a2c9         2          9             NaN            NaN   
10493       a2c10         2         10             NaN            NaN   
10704        a3c7         3          7             NaN            NaN   
10705        a3c8         3          8             NaN            NaN   

       Teenager(s)  Annual Self-Sufficiency Wage  
10276          NaN                     270980.15  
10277          NaN                     30162